In [75]:
from pathlib import Path
from rdkit import Chem
import pandas as pd
import json
import re

make dict of the properties that are returned from QFP

In [76]:
with open("../data/properties.json", "r") as f:
    properties_list = json.load(f)

property_dict = {prop["property_db_id"]: prop["name_of_property"] for prop in properties_list}
property_dict

{0: 'energy',
 1: 'atomization_energy',
 2: 'homo_lumo_gap',
 3: 'ionization_energy',
 4: 'electron_affinity',
 5: 'chemical_potential',
 6: 'molecular_dipole',
 7: 'molecular_dipole_norm',
 8: 'molecular_quadrupole',
 9: 'molecular_quadrupole_principal_invariant_2',
 10: 'molecular_quadrupole_principal_invariant_3',
 11: 'molecular_polarizability',
 12: 'molecular_polarizability_mean',
 13: 'molecular_polarizability_anisotropy',
 14: 'normal_modes',
 15: 'normal_mode_frequencies',
 16: 'infrared_intensity',
 17: 'enthalpy',
 18: 'gibbs_free_energy',
 19: 'heat_capacity',
 20: 'entropy',
 21: 'zero_point_energy',
 22: 'radius_of_gyration',
 23: 'molecular_volume',
 24: 'molecular_sasa',
 25: 'atomic_sasa',
 26: 'effective_coordination_number',
 27: 'partial_charge',
 28: 'atomic_fukui_minus',
 29: 'atomic_fukui_plus',
 30: 'atomic_dipole',
 31: 'atomic_dipole_norm',
 32: 'atomic_quadrupole',
 33: 'atomic_quadrupole_principal_invariant_2',
 34: 'atomic_quadrupole_principal_invariant_3',

The output files of the QFP program are given in the following format:

"example_input_{id}.json" provides a **list** of conformers. For each conformer, the xyz coordinates + bunch of properties

In [77]:
def create_molecule_df(molecule_data: list[dict], property_dict: dict[int, str]) -> pd.DataFrame:
    """
    Transform the data from QuantumFP into a pandas dataframe. The dataframe consists of all QM properties extracted from QFP for each conformer of the molecule.

    params: 
        molecule_data: A list containing a dict for each conformer of the molecule with its corresponding QM properties.

    return:
        a pd.DataFrame containing the QM properties of each conformer of the molecule.
    """
    pattern = re.compile(r"prop_id")

    molecule_dict = {}
    for idx, conformer in enumerate(molecule_data):
        # Get all QM properties of a conformer and assign the proper name of the property
        conformer_dict = {property_dict[int(k.split("_")[-1])]: v for k, v in conformer.items() if pattern.search(k)}

        # Add the SMILES representations to the dataframe
        conformer_dict.update({"original_smiles": conformer["original_smiles"], "output_smiles": conformer["output_smiles"]})

        # Add the dict of the conformer to the dict of the molecule
        molecule_dict[f"conformer {idx}"] = conformer_dict
    
    return pd.DataFrame(molecule_dict).T

In [82]:
def load_molecule_data(path: Path) -> dict:
    with open(path, "r") as f:
        data = json.load(f)
    
    return data

In [93]:
PATH = "../data/example_input_2.json"
PATH = Path("../data/QuantumFP/data.json")

molecule_data = load_molecule_data(PATH)
print(len(molecule_data))

9


In [94]:
molecule_df = create_molecule_df(molecule_data, property_dict)

In [97]:
molecule_df

,energy,atomization_energy,homo_lumo_gap,ionization_energy,electron_affinity,chemical_potential,molecular_dipole,molecular_dipole_norm,molecular_quadrupole,molecular_quadrupole_principal_invariant_2,...,bond_energy,bond_length,bond_stiffness,overlap_integral,nuclear_repulsion,atomic_charge_dipole_interaction,atomic_charge_quadrupole_interaction,atomic_dipole_dipole_interaction,original_smiles,output_smiles
conformer 0,-45.123968,7.142295,0.153754,0.464067,0.112448,-0.298219,"[-0.578864151645866, 0.04196664502259578, -0.2...",0.626192,"[[3.3118407123402904, -6.682083890270894, 3.33...",-68.875397,...,"[[1, 16, 0.21571048655281544], [1, 17, 0.21587...","[[1, 2, 2.893035945751123], [2, 3, 2.901110269...","[[1, 2, 0.6570187658208813], [2, 3, 0.63161608...","[[1, 2, 0.2049576440334363], [1, 3, 0.07401850...","[[1, 2, 6.584909873649758], [1, 3, 4.060250330...","[[1, 2, -0.00012856888065181262], [1, 3, -0.00...","[[1, 2, -0.0009316886501469151], [1, 3, -0.000...","[[2, 1, 0.00012755522769508597], [3, 1, 0.0010...",[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...
conformer 1,-45.124599,7.142925,0.156376,0.461382,0.105206,-0.291984,"[0.4795648850853666, -0.12746403434649733, -0....",0.496847,"[[-13.839117214586379, 2.211930001058607, 0.41...",-161.285209,...,"[[1, 16, 0.21423931389767858], [1, 17, 0.21634...","[[1, 2, 2.901072403180445], [2, 3, 2.901146135...","[[1, 2, 0.6315654603235277], [2, 3, 0.63150476...","[[1, 2, 0.18176248372920967], [1, 3, 0.0743443...","[[1, 2, 6.566668568187167], [1, 3, 4.013841730...","[[1, 2, -0.00037719393343426386], [1, 3, -0.00...","[[1, 2, -0.00016004900728609122], [1, 3, -0.00...","[[2, 1, 0.0004424247193082747], [3, 1, 0.00100...",[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...
conformer 2,-45.124015,7.142341,0.154069,0.464199,0.11238,-0.29822,"[-0.5884934314584283, 0.048328172325055097, -0...",0.636923,"[[3.518107199948586, -6.470727497371688, 3.215...",-66.244227,...,"[[1, 16, 0.21445595880341983], [1, 17, 0.21643...","[[1, 2, 2.900974250916873], [2, 3, 2.900923148...","[[1, 2, 0.6318773072585191], [2, 3, 0.63199865...","[[1, 2, 0.18167267882123106], [1, 3, 0.0667159...","[[1, 2, 6.566890746437762], [1, 3, 4.013697345...","[[1, 2, -0.0003515423852399467], [1, 3, -0.001...","[[1, 2, -0.0001314639387902998], [1, 3, -0.000...","[[2, 1, 0.0004138643925002562], [3, 1, 0.00102...",[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...
conformer 3,-45.122048,7.140374,0.155134,0.458234,0.104368,-0.290804,"[0.4201967232061874, -0.20091195520974015, 0.1...",0.499523,"[[-11.307701339155853, 1.8015321951259216, -1....",-123.078867,...,"[[1, 16, 0.21549621183270062], [1, 17, 0.21574...","[[1, 2, 2.8932393495896114], [2, 3, 2.90101511...","[[1, 2, 0.6565072326057139], [2, 3, 0.63177227...","[[1, 2, 0.24104740578405146], [1, 3, 0.0660275...","[[1, 2, 6.584446933746569], [1, 3, 4.060696620...","[[1, 2, -0.00017307380197363487], [1, 3, -0.00...","[[1, 2, -0.0009238978212165589], [1, 3, -0.000...","[[2, 1, 0.0001802611601004633], [3, 1, 0.00103...",[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...
conformer 4,-45.121996,7.140322,0.154799,0.457972,0.104287,-0.290671,"[-0.44241353358765756, -0.1707220685020225, -0...",0.51579,"[[-11.330171166030494, -1.5406416259528393, -1...",-122.599523,...,"[[1, 16, 0.2154963471908644], [1, 17, 0.215709...","[[1, 2, 2.893365395273712], [2, 3, 2.901142928...","[[1, 2, 0.6562950708064899], [2, 3, 0.63187418...","[[1, 2, 0.18146873923888068], [1, 3, 0.0666767...","[[1, 2, 6.584160090916574], [1, 3, 4.061494221...","[[1, 2, -0.00016144951188910858], [1, 3, -0.00...","[[1, 2, -0.0009577658226621131], [1, 3, -0.000...","[[2, 1, 0.00016525822042183012], [3, 1, 0.0010...",[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...
conformer 5,-45.123791,7.